# Import Libraries

In [2]:
!pip install hyperopt

In [3]:
# Import pandas for data wrangling
import pandas as pd

# Import numpy for scientific computations
import numpy as np

# Import XGBoost
import xgboost as xgb

# Import accuracy score
from sklearn.metrics import accuracy_score

# Import packages for hyperparameter tuning
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe

In [4]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


# Read Dataset

In [6]:
df = pd.read_csv(r"C:\Users\suraj\Downloads\archive (5)\Wholesale customers data.csv")

In [7]:
df

,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
0,2,3,12669,9656,7561,214,2674,1338
1,2,3,7057,9810,9568,1762,3293,1776
2,2,3,6353,8808,7684,2405,3516,7844
3,1,3,13265,1196,4221,6404,507,1788
4,2,3,22615,5410,7198,3915,1777,5185
...,...,...,...,...,...,...,...,...
435,1,3,29703,12051,16027,13135,182,2204
436,1,3,39228,1431,764,4510,93,2346
437,2,3,14531,15488,30243,437,14841,1867
438,1,3,10290,1981,2232,1038,168,2125


# Declare feature vector and target variable

In [9]:
X = df.drop('Channel', axis=1)

y = df['Channel']

In [10]:
X.head()

,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
0,3,12669,9656,7561,214,2674,1338
1,3,7057,9810,9568,1762,3293,1776
2,3,6353,8808,7684,2405,3516,7844
3,3,13265,1196,4221,6404,507,1788
4,3,22615,5410,7198,3915,1777,5185


In [11]:
y.head()

0    2
1    2
2    2
3    1
4    2
Name: Channel, dtype: int64

In [12]:
# convert labels into binary values

y[y == 2] = 0

y[y == 1] = 1

In [13]:
# again preview the y label

y.head()

0    0
1    0
2    0
3    1
4    0
Name: Channel, dtype: int64

# Split data into separate training and test set

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 0)

# Initialize domain space for of values

In [17]:
space={'max_depth': hp.quniform("max_depth", 3, 18, 1),
        'gamma': hp.uniform ('gamma', 1,9),
        'reg_alpha' : hp.quniform('reg_alpha', 40,180,1),
        'reg_lambda' : hp.uniform('reg_lambda', 0,1),
        'colsample_bytree' : hp.uniform('colsample_bytree', 0.5,1),
        'min_child_weight' : hp.quniform('min_child_weight', 0, 10, 1),
        'n_estimators': 180,
        'seed': 0
    }

# Define objective function

In [24]:
from hyperopt import STATUS_OK
from sklearn.metrics import accuracy_score
import xgboost as xgb

def objective(space):

    clf = xgb.XGBClassifier(
        n_estimators=int(space['n_estimators']),
        max_depth=int(space['max_depth']),
        gamma=space['gamma'],
        reg_alpha=float(space['reg_alpha']),
        min_child_weight=int(space['min_child_weight']),
        colsample_bytree=float(space['colsample_bytree']),
        eval_metric='auc'
    )

    clf.fit(
        X_train,
        y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    pred = clf.predict(X_test)

    accuracy = accuracy_score(y_test, pred)

    return {
        'loss': -accuracy,
        'status': STATUS_OK
    }

# Optimization algorithm 

In [26]:
trials = Trials()

best_hyperparams = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=100,
    trials=trials
)

print(best_hyperparams)

100%|█████████████████████████████████████████████| 100/100 [00:22<00:00,  4.47trial/s, best loss: -0.6515151515151515]
{'colsample_bytree': 0.87592084389152, 'gamma': 8.010381434027106, 'max_depth': 7.0, 'min_child_weight': 4.0, 'reg_alpha': 85.0, 'reg_lambda': 0.027508250795747147}


# Print Results

In [29]:
print("The best hyperparameters are : ","\n")
print(best_hyperparams)

The best hyperparameters are :  

{'colsample_bytree': 0.87592084389152, 'gamma': 8.010381434027106, 'max_depth': 7.0, 'min_child_weight': 4.0, 'reg_alpha': 85.0, 'reg_lambda': 0.027508250795747147}
